# Momants intentieclassificatie

Deze notebook traint en gebruikt een aparte SetFit-classifier voor zes vaste bezoekersintenties. Trainen gebruikt uitsluitend de handgeschreven voorbeelden uit `intentie_training.csv`.

## 1. Importeer de intentiemodule

In [ ]:
from pathlib import Path
import importlib
import sys

PROJECTMAP = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECTMAP) not in sys.path:
    sys.path.insert(0, str(PROJECTMAP))

import momants_intentie
importlib.reload(momants_intentie)

## 2. Stel de paden in

In [ ]:
TRAININGSDATA_PAD = PROJECTMAP / "intentie_training.csv"
MODEL_PAD = PROJECTMAP / "model" / "momants-intentie"
CSV_PAD = PROJECTMAP / "notebooks" / "public_main_membermessage_export_2026-08-31_150601.csv"
UITVOERMAP = PROJECTMAP / "resultaten"

## 3. Train het model eenmalig

Voer deze cel één keer uit. Bij latere classificaties kun je deze stap overslaan zolang de map `model/momants-intentie` bestaat.

In [ ]:
opgeslagen_model = momants_intentie.train_model(
    trainingsdata_pad=TRAININGSDATA_PAD,
    model_pad=MODEL_PAD,
)
print(f"Model opgeslagen in: {opgeslagen_model}")

## 4. Controleer de Momants-export zonder model

Deze stap leest alleen de toegestane velden en toont uitsluitend aantallen.

In [ ]:
data = momants_intentie.laad_momants_csv(CSV_PAD)
bezoekers = momants_intentie.selecteer_bezoekersberichten(data)
print(f"Berichtrijen: {len(data)}")
print(f"Bruikbare bezoekersberichten: {len(bezoekers)}")
print(f"Gesprekken: {bezoekers['conversation_id'].nunique()}")

## 5. Classificeer de intenties

Deze stap gebruikt het eerder getrainde lokale model en schrijft `resultaten/intenties_per_gesprek.csv`.

In [ ]:
intenties = momants_intentie.verwerk_csv(
    csv_pad=CSV_PAD,
    uitvoermap=UITVOERMAP,
    model_pad=MODEL_PAD,
    batchgrootte=32,
)
print(f"Klaar: {len(intenties)} gesprek-intenties gevonden.")
print(f"Bestand: {UITVOERMAP / 'intenties_per_gesprek.csv'}")
intenties.head(10)